# 23. Are the explanations stable? (research question 4)
Each year has 5 LSTMs (seeds). Do they tell the **same story** — do they rank the three markets in the same order? **H4:** the average Spearman correlation between the seeds' rankings is at least **0.8** in every window.

* **Needs:** `results/shap_by_seed.csv`, `results/shap_stability.csv`. **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
by_seed = read_csv(folder + 'results/shap_by_seed.csv')
by_seed.head(10)

,window,seed,domestic,US,euro
0,calm,1,0.565629,0.230639,0.203732
1,calm,2,0.531578,0.267240,0.201182
2,calm,3,0.491943,0.284231,0.223826
3,calm,4,0.514208,0.268685,0.217107
4,calm,5,0.521847,0.273905,0.204248
5,Global Financial Crisis,1,0.548306,0.215585,0.236108
6,Global Financial Crisis,2,0.545129,0.225570,0.229301
7,Global Financial Crisis,3,0.402147,0.379549,0.218304
8,Global Financial Crisis,4,0.522799,0.267262,0.209939
9,Global Financial Crisis,5,0.512584,0.284309,0.203107


The channel shares of the 5 seeds in the Global Financial Crisis:

In [3]:
gfc = by_seed[by_seed['window'] == 'Global Financial Crisis'].set_index('seed')[['domestic', 'US', 'euro']]
gfc.round(3)

,domestic,US,euro
seed,,,
1,0.548,0.216,0.236
2,0.545,0.226,0.229
3,0.402,0.380,0.218
4,0.523,0.267,0.210
5,0.513,0.284,0.203


## Spearman correlation = agreement between two rankings
`rank` gives each market its place (1 = smallest share, 3 = largest):

In [4]:
gfc.rank(axis=1)

,domestic,US,euro
seed,,,
1,3.0,1.0,2.0
2,3.0,1.0,2.0
3,3.0,2.0,1.0
4,3.0,2.0,1.0
5,3.0,2.0,1.0


Spearman's correlation is +1 when two seeds rank the markets identically and lower when they disagree. Seeds 1 and 2:

In [5]:
from scipy.stats import spearmanr
rho = spearmanr(gfc.loc[1], gfc.loc[2])[0]
print('Spearman, seed 1 vs seed 2:', rho)

Spearman, seed 1 vs seed 2: 1.0


All 10 pairs of seeds, with two loops (`a < b` so that each pair is counted once):

In [6]:
values = []
for a in [1, 2, 3, 4, 5]:
    for b in [1, 2, 3, 4, 5]:
        if a < b:
            rho = spearmanr(gfc.loc[a], gfc.loc[b])[0]
            values.append(rho)
            print('seed', a, 'vs seed', b, ':', round(rho, 2))
print('average:', round(sum(values) / len(values), 2))

seed 1 vs seed 2 : 1.0
seed 1 vs seed 3 : 0.5
seed 1 vs seed 4 : 0.5
seed 1 vs seed 5 : 0.5
seed 2 vs seed 3 : 0.5
seed 2 vs seed 4 : 0.5
seed 2 vs seed 5 : 0.5
seed 3 vs seed 4 : 1.0
seed 3 vs seed 5 : 1.0
seed 4 vs seed 5 : 1.0
average: 0.7


## Every window

In [7]:
for window in ['calm', 'Global Financial Crisis', 'Irish sovereign debt crisis', 'COVID-19', 'War / energy shock 2022']:
    table = by_seed[by_seed['window'] == window].set_index('seed')[['domestic', 'US', 'euro']]
    values = []
    for a in [1, 2, 3, 4, 5]:
        for b in [1, 2, 3, 4, 5]:
            if a < b:
                values.append(spearmanr(table.loc[a], table.loc[b])[0])
    print(window, ': average Spearman =', round(sum(values) / len(values), 2))

calm : average Spearman = 1.0
Global Financial Crisis : average Spearman = 0.7
Irish sovereign debt crisis : average Spearman = 0.7
COVID-19 : average Spearman = 0.8
War / energy shock 2022 : average Spearman = 1.0


In [8]:
saved = read_csv(folder + 'results/shap_stability.csv', index_col=0)
saved.round(2)

,mean_spearman_channels,min_spearman_channels,mean_spearman_6_inputs
window,,,
calm,1.0,1.0,0.81
Global Financial Crisis,0.7,0.5,0.62
Irish sovereign debt crisis,0.7,0.5,0.83
COVID-19,0.8,0.5,0.82
War / energy shock 2022,1.0,1.0,0.87


In [9]:
if (saved['mean_spearman_channels'] >= 0.8).all():
    print('H4 supported')
else:
    print('H4 NOT supported: in some windows the seeds disagree')

H4 NOT supported: in some windows the seeds disagree


Fully stable in calm times and 2022 (1.0), less so in the GFC and the Irish crisis (0.7). With only three markets one swapped pair already lowers Spearman to 0.5, so the measure is coarse; the column with 6 inputs gives a finer picture. The explanations were least stable in the GFC — the window where the model was also least accurate.